# Q2 — Unsupervised Learning: Customer Segmentation

**Objective:** Segment customers using K-Means clustering, validate with PCA, and derive actionable business profiles.

**Dataset:**  — 500 customers, 6 behavioural/demographic features, no target column.

## Task 1 — Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

df = pd.read_csv("../data/q2_customers.csv")

print(f"Dataset shape: {df.shape}")
print("
Data types:")
print(df.dtypes)
print("
Missing values:")
print(df.isnull().sum())
print("
First five rows:")
display(df.head())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print(f"
Scaling complete. X_scaled shape: {X_scaled.shape}")
print("Sample scaled values (first 3 rows):")
print(np.round(X_scaled[:3], 2))

### Why Scaling is Essential Before K-Means

K-Means assigns each data point to the nearest centroid using **Euclidean distance**. When features are measured on very different scales —  ranges from 5,038 to 119,757 while  ranges from 1 to 19 — a difference of 50,000 in spend completely dominates the distance calculation, making the algorithm cluster almost exclusively on spend while ignoring visit frequency, recency, and all other signals.

 transforms each feature to zero mean and unit variance, placing all six variables on equal footing so that age, spending, frequency, basket size, recency, and category breadth each contribute equally to the notion of customer similarity. Since K-Means has no train-test split, the scaler is legitimately fit on the full 500-row dataset with no data leakage concern.

## Task 2 — Choosing K: Elbow Method

In [ ]:
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

print("WCSS for K = 1 to 10:")
for i, w in enumerate(wcss, 1):
    print(f"  K={i:>2}  WCSS = {w:>7.1f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_range), wcss, marker="o", color="steelblue",
        linewidth=2.5, markersize=8, markerfacecolor="white", markeredgewidth=2)
ax.axvline(x=4, color="crimson", linestyle="--", linewidth=1.8, label="Elbow at K = 4")
ax.set_xlabel("Number of Clusters (K)", fontsize=12)
ax.set_ylabel("WCSS (Within-Cluster Sum of Squares)", fontsize=12)
ax.set_title("Elbow Method — Optimal K Selection", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_xticks(list(K_range))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Elbow Point Analysis — Choosing K = 4

The WCSS drops steeply from K=1 through K=4 and then flattens. The marginal gain at each step:

| K | WCSS | Drop from K−1 |
|---|------|---------------|
| 1 | 3000.0 | — |
| 2 | 969.0 | −2031.0 |
| 3 | 561.3 | −407.7 |
| **4** | **444.9** | **−116.4** |
| 5 | 402.4 | −42.5 |
| 6 | 370.4 | −32.0 |

The inflection occurs at **K=4**: moving from K=4 to K=5 yields only 42.5 units of additional improvement versus 116.4 from K=3 to K=4 — a 64% collapse in marginal gain. This is the textbook elbow signal. K=4 also maps to four interpretable, actionable marketing personas, making it the practically optimal choice.

## Task 3 — K-Means Clustering with K = 4

In [ ]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_scaled)

print("Cluster assignment complete.")
print("Cluster sizes:")
for c, cnt in df["cluster"].value_counts().sort_index().items():
    print(f"  Cluster {c}: {cnt:>3} customers")

feature_cols = [c for c in df.columns if c != "cluster"]
centroids = pd.DataFrame(
    scaler.inverse_transform(km_final.cluster_centers_),
    columns=feature_cols
).round(1)
centroids.index = [f"Cluster {i}" for i in range(4)]

print("
Cluster Centroids (original feature scale):")
display(centroids)

### Business Interpretation of Each Cluster

| Cluster | Size | Persona | Key Defining Characteristics |
|---------|------|---------|------------------------------|
| **0** | 170 | Young Budget Shoppers | Avg age 25, spend ~14,847/yr, 1 visit/month, last seen 9 days ago — active but very low-value |
| **1** | 80 | Churned High-Value Seniors | Avg age 57, historically high spend ~89,814/yr, but **148 days since last visit** — high churn risk |
| **2** | 165 | Engaged Mid-Spenders | Avg age 40, spend ~43,341/yr, 3 visits/month, last seen 35 days — reliable core segment |
| **3** | 85 | Active Premium Customers | Avg age 57, spend ~89,036/yr, 7 visits/month, last seen 65 days — high-value and actively engaged |

**Actionable Business Implications:**
- **Cluster 0 — Young Budget Shoppers:** Bundle deals and entry-level loyalty tiers to grow basket size and frequency. These are customers to develop over the long term.
- **Cluster 1 — Churned High-Value:** Highest-priority win-back campaign. Personalised outreach referencing past purchase history with an exclusive incentive can recover significant annual revenue.
- **Cluster 2 — Engaged Mid-Spenders:** Upsell into premium categories. Consistent engagement makes them ideal for cross-sell promotions and loyalty programme upgrades.
- **Cluster 3 — Active Premium:** Focus on retention with VIP perks, early access, and personalised service to prevent drift toward the churned Cluster 1 profile.

## Task 4 — Dimensionality Reduction with PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

evr = pca.explained_variance_ratio_
print("Explained Variance Ratios:")
print(f"  PC1: {evr[0]*100:.2f}%")
print(f"  PC2: {evr[1]*100:.2f}%")
print(f"  Combined (2 components): {sum(evr)*100:.2f}%")

loadings = pd.DataFrame(
    pca.components_,
    columns=feature_cols,
    index=["PC1", "PC2"]
).round(4)

print("
Feature Loadings (components matrix):")
display(loadings)

### Interpretation of PC1 and PC2

**PC1 — Overall Customer Value / Engagement Axis (83.56% of variance)**

PC1 carries nearly 84% of all variance — the dominant axis by a wide margin. Five of six features load with nearly equal positive magnitudes (0.38–0.42): age, annual spend, basket size, days since last visit, and num_categories_purchased all increase together, confirming these behaviours are highly co-linear.  loads negatively at −0.41, reflecting that the most frequent visitors in this dataset tend to be younger, lower-spending customers in Cluster 0. PC1 effectively ranks customers from low-value (left) to high-value (right).

**PC2 — Recency Axis (5.57% of variance)**

PC2 is dominated almost entirely by  (loading 0.91), with all other features contributing near zero. This axis captures **recency independently of overall value**. Clusters 1 and 3 are nearly indistinguishable on PC1 — both old and high-spending — but PC2 cleanly separates them: Cluster 1 customers were last seen 148 days ago versus 65 days for Cluster 3. This validates recency as the single most critical variable for identifying at-risk customers within the high-value tier.

## Task 5 — Cluster Visualisation

In [ ]:
cluster_labels = {
    0: "Cluster 0: Young Budget Shoppers",
    1: "Cluster 1: Churned High-Value Seniors",
    2: "Cluster 2: Engaged Mid-Spenders",
    3: "Cluster 3: Active Premium Customers"
}
colors = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12"]

fig, ax = plt.subplots(figsize=(10, 7))

for c in range(4):
    mask = df["cluster"] == c
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colors[c],
        label=cluster_labels[c],
        alpha=0.75,
        edgecolors="white",
        linewidths=0.4,
        s=65
    )

centroids_pca = pca.transform(km_final.cluster_centers_)
ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    marker="X", c="black", s=220, zorder=5, label="Centroids"
)

ax.set_xlabel(f"PC1 — Overall Value Axis ({evr[0]*100:.1f}% variance explained)", fontsize=11)
ax.set_ylabel(f"PC2 — Recency Axis ({evr[1]*100:.1f}% variance explained)", fontsize=11)
ax.set_title("Customer Segments — PCA Projection (K = 4 Clusters)", fontsize=13, fontweight="bold")
ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()